# Underwater Waste Detection — Baseline Training

This notebook:
1. Converts TrashCan 1.0 COCO annotations → YOLO format
2. Trains **YOLOv8n** (fast) and optionally **YOLOv8m** (accurate)
3. Plots training curves (loss, mAP)
4. Saves best weights to Google Drive

**Runtime requirement:** GPU (T4 or better). Change Runtime → Change runtime type → GPU.

In [ ]:
# ── Cell 1: Verify GPU & install packages ─────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch runtime to GPU!')

!pip install -q ultralytics pycocotools

In [ ]:
# ── Cell 2: Mount Drive & set paths ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE       = Path('/content/drive/MyDrive/underwater_waste_detection')
RAW_DATA_DIR     = DRIVE_BASE / 'trashcan_raw' / 'extracted'
YOLO_DATASET_DIR = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR      = DRIVE_BASE / 'weights'

for d in [YOLO_DATASET_DIR, WEIGHTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clone repo (update URL after pushing to GitHub)
REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 3: COCO → YOLO annotation conversion ─────────────────────────────
import json
import shutil
import random

# TrashCan fine-grained class → macro-category mapping
MACRO_MAP = {
    # trash = 0
    'plastic_bag': 0, 'plastic_bottle': 0, 'milk_jug': 0, 'paper_bag': 0,
    'cup': 0, 'container': 0, 'bottle_cap': 0, 'net': 0, 'pipe': 0,
    'plastic_tray': 0, 'rubber_glove': 0, 'rubber_boot': 0, 'styrofoam': 0,
    'unknown_instance': 0, 'unlabeled_trash': 0,
    # bio = 1
    'fish': 1, 'eel': 1, 'crab': 1, 'starfish': 1, 'nudibranch': 1,
    'coral': 1, 'shell': 1, 'lobster': 1, 'octopus': 1,
    'sea_cucumber': 1, 'shrimp': 1, 'jellyfish': 1,
    # rov = 2
    'rov': 2,
}

def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    return cx, cy, w / img_w, h / img_h

def convert_split(coco_json_path, images_src_dir, yolo_images_dir, yolo_labels_dir):
    yolo_images_dir.mkdir(parents=True, exist_ok=True)
    yolo_labels_dir.mkdir(parents=True, exist_ok=True)

    with open(coco_json_path) as f:
        data = json.load(f)

    img_meta   = {img['id']: img for img in data['images']}
    cat_names  = {cat['id']: cat['name'] for cat in data['categories']}

    anns_by_img = {}
    for ann in data['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)

    converted, skipped = 0, 0
    for img_id, meta in img_meta.items():
        src_img = images_src_dir / meta['file_name']
        if not src_img.exists():
            skipped += 1
            continue

        shutil.copy2(src_img, yolo_images_dir / meta['file_name'])

        lines = []
        for ann in anns_by_img.get(img_id, []):
            cat = cat_names.get(ann['category_id'], '')
            cls_id = MACRO_MAP.get(cat, -1)
            if cls_id < 0 or not ann.get('bbox'):
                continue
            cx, cy, w_n, h_n = coco_bbox_to_yolo(ann['bbox'], meta['width'], meta['height'])
            if 0 < w_n <= 1 and 0 < h_n <= 1:
                lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w_n:.6f} {h_n:.6f}')

        label_file = yolo_labels_dir / (Path(meta['file_name']).stem + '.txt')
        label_file.write_text('\n'.join(lines))
        converted += 1

    print(f'  Converted: {converted} images, Skipped: {skipped}')
    return converted

# Find annotation files
ann_files = sorted(RAW_DATA_DIR.rglob('*.json'))
print(f'Found annotation files:')
for f in ann_files:
    print(f'  {f.name}')

# TrashCan has train/val splits already; find them
train_ann = next((f for f in ann_files if 'train' in f.name.lower()), None)
val_ann   = next((f for f in ann_files if 'val' in f.name.lower()), None)

if train_ann:
    images_src = train_ann.parent / 'images' if (train_ann.parent / 'images').exists() else RAW_DATA_DIR / 'images'
    print(f'\nConverting training split...')
    convert_split(train_ann, images_src,
                  YOLO_DATASET_DIR / 'images' / 'train',
                  YOLO_DATASET_DIR / 'labels' / 'train')

if val_ann:
    print('Converting validation split...')
    convert_split(val_ann, images_src,
                  YOLO_DATASET_DIR / 'images' / 'val',
                  YOLO_DATASET_DIR / 'labels' / 'val')

print('\nConversion complete!')

In [ ]:
# ── Cell 4: Write dataset YAML ────────────────────────────────────────────
import yaml

dataset_config = {
    'path': str(YOLO_DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {0: 'trash', 1: 'bio', 2: 'rov'}
}

yaml_path = Path('/content/underwater-waste-detection/configs/dataset_colab.yaml')
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f'Dataset config written: {yaml_path}')
print(yaml_path.read_text())

In [ ]:
# ── Cell 5: Verify dataset structure ─────────────────────────────────────
for split in ['train', 'val']:
    n_imgs = len(list((YOLO_DATASET_DIR / 'images' / split).glob('*')))
    n_lbls = len(list((YOLO_DATASET_DIR / 'labels' / split).glob('*.txt')))
    print(f'{split:6s}: {n_imgs:5d} images, {n_lbls:5d} label files')

# Quick sanity check: sample label
sample_labels = list((YOLO_DATASET_DIR / 'labels' / 'train').glob('*.txt'))[:3]
print('\nSample label files:')
for lbl in sample_labels:
    content = lbl.read_text().strip()
    print(f'  {lbl.name}: {content[:80]}...' if len(content) > 80 else f'  {lbl.name}: {content}')

In [ ]:
# ── Cell 6: Train YOLOv8n (baseline — nano, fast) ─────────────────────────
from ultralytics import YOLO

# YOLOv8n: ~3.2M params, ~8.7 GFLOPs — fast, good for initial baseline
detector_nano = YOLO('yolov8n.pt')

training_results_nano = detector_nano.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,            # reduce to 8 if OOM
    device=0,
    project=str(DRIVE_BASE / 'runs'),
    name='yolov8n_baseline',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    save_period=20,
    verbose=True,
)

print('\nYOLOv8n training complete!')
print(f'Best weights: {DRIVE_BASE}/runs/yolov8n_baseline/weights/best.pt')

In [ ]:
# ── Cell 7: (Optional) Train YOLOv8m (medium — better accuracy) ───────────
# Uncomment to run after nano baseline is complete

# detector_medium = YOLO('yolov8m.pt')
# training_results_medium = detector_medium.train(
#     data=str(yaml_path),
#     epochs=100,
#     imgsz=640,
#     batch=8,           # medium needs more VRAM
#     device=0,
#     project=str(DRIVE_BASE / 'runs'),
#     name='yolov8m_baseline',
#     optimizer='AdamW',
#     lr0=0.001,
#     lrf=0.01,
#     warmup_epochs=3,
#     mosaic=1.0,
#     mixup=0.1,
# )

In [ ]:
# ── Cell 8: Plot training curves ──────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

results_csv = DRIVE_BASE / 'runs' / 'yolov8n_baseline' / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # strip whitespace from column names

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    metrics_to_plot = [
        ('train/box_loss', 'Train Box Loss', axes[0, 0]),
        ('train/cls_loss', 'Train Class Loss', axes[0, 1]),
        ('train/dfl_loss', 'Train DFL Loss', axes[0, 2]),
        ('metrics/mAP50(B)', 'mAP@0.5', axes[1, 0]),
        ('metrics/mAP50-95(B)', 'mAP@0.5:0.95', axes[1, 1]),
        ('metrics/precision(B)', 'Precision', axes[1, 2]),
    ]

    for col, title, ax in metrics_to_plot:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], linewidth=2, color='#2980b9')
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'{col}\nnot found', ha='center', va='center')

    plt.suptitle('YOLOv8n Training Curves — Underwater Waste Detection', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plot_path = DRIVE_BASE / 'runs' / 'yolov8n_baseline' / 'training_curves.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Training curves saved: {plot_path}')
else:
    print(f'results.csv not found at {results_csv}. Run training first.')

In [ ]:
# ── Cell 9: Copy best weights to Drive weights folder ─────────────────────
import shutil

best_weights_src = DRIVE_BASE / 'runs' / 'yolov8n_baseline' / 'weights' / 'best.pt'
best_weights_dst = WEIGHTS_DIR / 'yolov8n_best.pt'

if best_weights_src.exists():
    shutil.copy2(best_weights_src, best_weights_dst)
    print(f'Best weights saved: {best_weights_dst}')
else:
    print('best.pt not found — training may not have completed.')

print('\nNext step → Run 03_evaluation.ipynb to compute metrics and visualize results.')